<a href="https://colab.research.google.com/github/adityaabhishek001/Image_Deblurring_WiDS_using_DeepLearning/blob/Week-2/CNN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as f

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [11]:
class CNN(nn.Module):
    def __init__(self,in_chan,out_classes,o1,o2):
        super(CNN,self).__init__()
        self.conv1=nn.Conv2d(in_channels=in_chan,out_channels=o1,kernel_size=(3,3),stride=(1,1),padding=(1,1))
        self.pool=nn.MaxPool2d(kernel_size=(2,2),stride=(2,2))
        self.conv2=nn.Conv2d(in_channels=o1,out_channels=o2,kernel_size=(3,3),stride=(1,1),padding=(1,1))
        self.fc3=nn.Linear(32*8*8,128)
        self.fc4=nn.Linear(128,10)
        self.bn1=nn.BatchNorm2d(o1)
        self.bn2=nn.BatchNorm2d(o2)
        self.bn3=nn.BatchNorm1d(128)
        self.flatten = nn.Flatten()


    def forward(self,x):
        x=f.relu(self.bn1(self.conv1(x)))
        x=self.pool(x)
        x=f.relu(self.bn2(self.conv2(x)))
        x=self.pool(x)
        x=self.flatten(x)
        x=f.relu(self.bn3(self.fc3(x)))
        x=self.fc4(x)
        return x

In [5]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

train_data=datasets.CIFAR10(root='./data',train=True,download=True,transform=transforms.ToTensor())
test_data=datasets.CIFAR10(root='./data',train=False,download=True,transform=transforms.ToTensor())

100%|██████████| 170M/170M [00:03<00:00, 48.9MB/s]


In [6]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)

In [15]:
in_chan=3
out_classes=10
o1=16
o2=32
learning_rate=0.001
num_epochs=10
model=CNN(in_chan,out_classes,o1,o2).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [16]:
model.train()
for epoch in range(num_epochs):
    i=0
    for images,labels in train_loader:

        images=images.to(device)
        labels=labels.to(device)

        output=model(images)
        loss=criterion(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/10], Loss: 0.8892
Epoch [2/10], Loss: 0.9308
Epoch [3/10], Loss: 0.8278
Epoch [4/10], Loss: 0.6720
Epoch [5/10], Loss: 0.9158
Epoch [6/10], Loss: 1.0282
Epoch [7/10], Loss: 0.8703
Epoch [8/10], Loss: 0.5314
Epoch [9/10], Loss: 0.4711
Epoch [10/10], Loss: 0.3243


In [17]:
correct=0
total=0
model.eval()
for images,labels in test_loader:

    images=images.to(device)
    labels=labels.to(device)

    outputs=model(images)
    loss=criterion(outputs,labels)
    correct+=(outputs.argmax(1)==labels).sum().item()
    total+=labels.size(0)
print(f'Test Accuracy: {100*correct/total:.2f}%')

Test Accuracy: 69.18%
